# 실습03_피아노
- MediaPipe Hands를 사용하여 손가락 움직임을 감지하고, 손가락 끝이 특정 위치(건반)에 닿으면 해당 건반의 음계를 출력하는 가상 피아노(키보드)를 구현
- OpenCV를 사용하여 화면에 건반을 표시하고, 손가락이 건반을 터치하면 해당 음계를 화면에 출력

In [1]:
import cv2
import mediapipe as mp
import numpy as np

from pathlib import Path
import urllib.request
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [3]:
# HandLandmarker 모델 준비
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

HAND_LANDMARKER_MODEL = MODEL_DIR / 'hand_landmarker.task'
HAND_LANDMARKER_MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'


def download_model(model_path, model_url):
    if model_path.exists():
        return
    print(f'모델 다운로드 중: {model_path}')
    urllib.request.urlretrieve(model_url, model_path)


download_model(HAND_LANDMARKER_MODEL, HAND_LANDMARKER_MODEL_URL)

BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
mp_hands = vision.HandLandmarksConnections
mp_drawing = vision.drawing_utils


def create_hand_landmarker(num_hands=2):
    # 손 검출/추적 옵션 설정
    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(HAND_LANDMARKER_MODEL)),
        running_mode=VisionRunningMode.VIDEO,
        num_hands=num_hands,
        min_hand_detection_confidence=0.5, # 손이 잘 안 잡히면 낮추고, 배경을 손으로 잘못 잡으면 높인다.
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )
    return HandLandmarker.create_from_options(options)


def to_mp_image(frame):
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)


def landmark_to_pixel(landmark, frame_shape):
    h, w = frame_shape[:2]
    return int(landmark.x * w), int(landmark.y * h)


In [4]:
# 건반 정의: x, y, 너비, 높이
keys = [(100, 350, 50, 100), (150, 350, 50, 100), (200, 350, 50, 100), (250, 350, 50, 100), (300, 350, 50, 100)]

# 건반별 음계
key_scales = ['C', 'D', 'E', 'F', 'G']

# 건반 색상
key_colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 165, 0)]

# 프레임별 건반 상태
key_status = [False] * len(keys)

In [ ]:
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_hand_landmarker(num_hands=2) as hand_landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        frame = cv2.flip(frame, 1)
        key_status = [False] * len(keys)

        mp_image = to_mp_image(frame)
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        # 검지 끝의 건반 위치 확인
        if result.hand_landmarks:
            for hand_landmarks in result.hand_landmarks:
                # TODO: 8번 랜드마크는 검지 끝
                x, y = landmark_to_pixel(hand_landmarks[8], frame.shape)
                cv2.circle(frame, (x, y), 8, (0, 255, 255), -1)

                for i, key in enumerate(keys):
                    key_x, key_y, key_w, key_h = key
                    # TODO: 검지 끝이 건반 안에 있는지 확인
                    if key_x <= x <= key_x + key_w and key_y <= y <= key_y + key_h:
                        key_status[i] = True

        # 손 검출 후 건반 그리기
        # 건반은 손 검출 후에 그려야 눌린 상태를 반영할 수 있음
        for i, key in enumerate(keys):
            key_x, key_y, key_w, key_h = key
            thickness = cv2.FILLED if key_status[i] else 2

            # TODO: 건반 그리기
            cv2.rectangle(
                frame,
                (key_x, key_y),                     # 사각형 왼쪽 위 좌표
                (key_x + key_w, key_y + key_h),     # 사각형의 오른쪽 아래 좌표
                key_colors[i],                      # i번째 건반의 색상
                thickness=                          # 채우기 또는 테두리 두께 
            )

            # TODO: 음계 텍스트 출력
            cv2.putText(
                frame,
                key_scales[i],
                (key_x + 10, key_y + 70),           # 글자가 시작될 위치
                cv2.FONT_HERSHEY_SIMPLEX,           # 글꼴
                1,                                  # 글자 크기
                (255, 255, 255),                    # 흰색
                2                                   # 글자 두께
            )

        cv2.imshow('Virtual Piano Scales', frame)
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

- 추가 기능 실습
    - 음계 소리 추가
    - 건반 개수 추가